# Retrieval Augmented Generation with a Graph Database

This notebook shows how to use LLMs in combination with [Neo4j](https://neo4j.com/), a graph database, to perform Retrieval Augmented Generation (RAG).

### Why use RAG?

If you want to use LLMs to generate answers based on your own content or knowledge base, instead of providing large context when prompting the model, you can fetch the relevant information in a database and use this information to generate a response. 

This allows you to:
- Reduce hallucinations
- Provide relevant, up to date information to your users
- Leverage your own content/knowledge base

### Why use a graph database?

If you have data where relationships between data points are important and you might want to leverage that, then it might be worth considering graph databases instead of traditional relational databases.

Graph databases are good to address the following:
- Navigating deep hierarchies
- Finding hidden connections between items
- Discovering relationships between items

### Use cases 

Graph databases are particularly relevant for recommendation systems, network relationships or analysing correlation between data points.  

Example use cases for RAG with graph databases include:
- Recommendation chatbot
- AI-augmented CRM 
- Tool to analyse customer behavior with natural language

Depending on your use case, you can assess whether using a graph database makes sense. 

In this notebook, we will build a **candidate recommendation chatbot**, with a graph database that contains candidate data.


## Setup

We will start by installing and importing the relevant libraries.  

Make sure you have your OpenAI account set up and you have your OpenAI API key handy. 

In [1]:
# # Optional: run to install the libraries locally if you haven't already 
# !pip3 install langchain
# !pip3 install openai
# !pip3 install neo4j

In [2]:
import os
import json 
import pandas as pd

In [ ]:
# Optional: run to load environment variables from a .env file.
# This is not required if you have exported your env variables in another way or if you set it manually
!pip3 install python-dotenv
from dotenv import load_dotenv
load_dotenv()

# Set the OpenAI API key env variable manually
# 
os.environ["OPENAI_API_KEY"] = ""

# print(os.environ["OPENAI_API_KEY"])

## Dataset

We will use a dataset that was created from a relational database and converted to a json format, creating relationships between entities with the completions API.

We will then load this data into the graph db to be able to query it.

### Loading dataset

In [4]:
# Loading a json dataset from a file
file_path = 'data/candidates_20250430-162540.json'

with open(file_path, 'r') as file:
    jsonData = json.load(file)

In [5]:
df =  pd.read_json(file_path)
df.head()

,candidate_id,name,relationship,entity_type,entity_value,TITLE,SUMMARY
0,1,Mahir Jain,hasEducationLevel,education,Masters,Data Scientist with Analytics Experience,"Data scientist with expertise in Python, SQL, ..."
1,1,Mahir Jain,studiedAt,university,University of Washington,Data Scientist with Analytics Experience,"Data scientist with expertise in Python, SQL, ..."
2,1,Mahir Jain,hasSkill,skill,Statistical Data Analysis,Data Scientist with Analytics Experience,"Data scientist with expertise in Python, SQL, ..."
3,1,Mahir Jain,hasSkill,skill,Data Visualization,Data Scientist with Analytics Experience,"Data scientist with expertise in Python, SQL, ..."
4,1,Mahir Jain,hasSkill,skill,Machine Learning,Data Scientist with Analytics Experience,"Data scientist with expertise in Python, SQL, ..."


### Connecting to db

In [6]:
from neo4j import GraphDatabase
# DB credentials
url = "bolt://localhost:7687"
username ="neo4j"
password = "thisismypassword"

URI = url
AUTH = (username, password)

with GraphDatabase.driver(URI, auth=AUTH) as driver:
    driver.verify_connectivity()
    print("Connection established.")

Connection established.


In [7]:
from langchain.graphs import Neo4jGraph

graph = Neo4jGraph(
    url=url, 
    username=username, 
    password=password
)

/var/folders/_1/f4gv917x3jv_z8t21wgjlcrw0000gn/T/ipykernel_34869/860924429.py:3: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-neo4j package and should be used instead. To use it run `pip install -U :class:`~langchain-neo4j` and import as `from :class:`~langchain_neo4j import Neo4jGraph``.
  graph = Neo4jGraph(


### Importing data

In [8]:
def sanitize(text):
    text = str(text).replace("'", "").replace('"', '').replace("{", "").replace("}", "")
    return text

i = 1
for obj in jsonData:
    print(f"{i}. {obj['candidate_id']} - {obj['relationship']} -> {obj['entity_value']}")
    i += 1
    query = f'''
        // MERGE candidate node by candidate id
        MERGE (candidate:Candidate {{id: {obj['candidate_id']}}})
        ON CREATE SET candidate.name = "{sanitize(obj['name'])}",
                      candidate.title = "{sanitize(obj['TITLE'])}",
                      candidate.summary = "{sanitize(obj['SUMMARY'])}"
        ON MATCH SET candidate.name = "{sanitize(obj['name'])}",
                     candidate.title = "{sanitize(obj['TITLE'])}",
                     candidate.summary = "{sanitize(obj['SUMMARY'])}"
        
        // MERGE entity node by its type and value
        MERGE (entity:{obj['entity_type']} {{value: "{sanitize(obj['entity_value'])}"}})
        ON CREATE SET entity.value = "{sanitize(obj['entity_value'])}"
        ON MATCH SET entity.value = "{sanitize(obj['entity_value'])}"
        
        // Create (or merge) the relationship from candidate to entity
        MERGE (candidate)-[r:{obj['relationship']}]->(entity)
        // If the relationship has properties that should be updated, add ON CREATE/ON MATCH SET here.
        // For example, to update a timestamp or counter:
        // ON CREATE SET r.createdAt = timestamp()
        // ON MATCH SET r.updatedAt = timestamp()
        '''
    graph.query(query)


1. 1 - hasEducationLevel -> Masters
2. 1 - studiedAt -> University of Washington
3. 1 - hasSkill -> Statistical Data Analysis
4. 1 - hasSkill -> Data Visualization
5. 1 - hasSkill -> Machine Learning
6. 1 - hasSkill -> A/B testing
7. 1 - hasProgrammingExperience -> Python
8. 1 - hasProgrammingExperience -> R
9. 1 - hasDataQueryExperience -> SQL
10. 1 - hasJobTitle -> Data Scientist Intern
11. 1 - hasJobTitle -> Senior Data Analyst
12. 1 - workedAt -> Cox Communications
13. 1 - workedAt -> Cisco Systems, Inc.
14. 1 - workedAt -> IBM
15. 1 - certifiedIn -> AWS Cloud Practitioner
16. 1 - certifiedIn -> Tableau Desktop Specialist
17. 1 - hasProject -> Social Media Activity Simulation
18. 1 - hasProject -> YouTube Content Virality Analysis
19. 1 - hasMLExperience -> LSTM
20. 1 - hasExperimentationExperience -> A/B testing
21. 1 - hasCausalInferenceExperience -> Causal Inference
22. 2 - hasEducationLevel -> Masters
23. 2 - studiedAt -> University of Washington
24. 2 - hasSkill -> SQL
25. 2 -

In [15]:
graph.query('MATCH (n) RETURN count(n) AS totalNodes;')

[{'totalNodes': 34}]

In [16]:
graph.query('MATCH ()-[r]->() RETURN count(r) AS totalRelationships;')

[{'totalRelationships': 40}]

In [17]:
graph.query('MATCH (c:Candidate)-[r]->(e) RETURN c.id, c.name, type(r) AS relationship, e.value LIMIT 25;')

[{'c.id': 1,
  'c.name': 'Hongfan (Louise) Lu',
  'relationship': 'hasSkill',
  'e.value': 'Python'},
 {'c.id': 1,
  'c.name': 'Hongfan (Louise) Lu',
  'relationship': 'hasSkill',
  'e.value': 'SQL'},
 {'c.id': 1,
  'c.name': 'Hongfan (Louise) Lu',
  'relationship': 'hasSkill',
  'e.value': 'Tableau'},
 {'c.id': 1,
  'c.name': 'Hongfan (Louise) Lu',
  'relationship': 'hasJobTitle',
  'e.value': 'Product Data Science Intern'},
 {'c.id': 1,
  'c.name': 'Hongfan (Louise) Lu',
  'relationship': 'hasJobTitle',
  'e.value': 'Business Development Data Analyst'},
 {'c.id': 1,
  'c.name': 'Hongfan (Louise) Lu',
  'relationship': 'workedAt',
  'e.value': 'PACCAR'},
 {'c.id': 1,
  'c.name': 'Hongfan (Louise) Lu',
  'relationship': 'workedAt',
  'e.value': 'Seraph Consulting'},
 {'c.id': 1,
  'c.name': 'Hongfan (Louise) Lu',
  'relationship': 'hasIndustry',
  'e.value': 'Automotive'},
 {'c.id': 1,
  'c.name': 'Hongfan (Louise) Lu',
  'relationship': 'hasEducationLevel',
  'e.value': 'Master of Sci

## Querying the database

### Creating vector indexes

In order to efficiently search our database for terms closely related to user queries, we need to use embeddings. To do this, we will create vector indexes on each type of property.

We will be using the OpenAIEmbeddings Langchain utility. It's important to note that Langchain adds a pre-processing step, so the embeddings will slightly differ from those generated directly with the OpenAI embeddings API.

In [18]:
from langchain.vectorstores.neo4j_vector import Neo4jVector
from langchain.embeddings.openai import OpenAIEmbeddings
embeddings_model = "text-embedding-3-small"

In [19]:
vector_index = Neo4jVector.from_existing_graph(
    OpenAIEmbeddings(model=embeddings_model),
    url=url,
    username=username,
    password=password,
    index_name='candidates',
    node_label="Candidate",
    text_node_properties=['name', 'title'],
    embedding_node_property='embedding'
)

/var/folders/_1/f4gv917x3jv_z8t21wgjlcrw0000gn/T/ipykernel_19141/272743987.py:2: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  OpenAIEmbeddings(model=embeddings_model),


In [20]:
def embed_entities(entity_type):
    vector_index = Neo4jVector.from_existing_graph(
        OpenAIEmbeddings(model=embeddings_model),
        url=url,
        username=username,
        password=password,
        index_name=entity_type,
        node_label=entity_type,
        text_node_properties=['value'],
        embedding_node_property='embedding'
    )
    
entities_list = df['entity_type'].unique()

for t in entities_list:
    embed_entities(t)

### Extracting entities from the prompt

However, there is little added value here compared to just writing the Cypher queries ourselves, and it is prone to error.

Indeed, asking an LLM to generate a Cypher query directly might result in the wrong parameters being used, whether it's the entity type or the relationship type, as is the case above.

We will instead use LLMs to decide what to search for, and then generate the corresponding Cypher queries using templates.

For this purpose, we will instruct our model to find relevant entities in the user prompt that can be used to query our database.

In [21]:
entity_types = {
    "skill": "A technical skill possessed by the candidate, e.g., 'Python', 'SQL', 'Tableau'",
    "job_title": "A specific role or position held by the candidate, e.g., 'Data Scientist', 'Software Engineer'",
    "company": "An organization where the candidate has worked",
    "industry": "The domain or sector associated with a company or job, e.g., 'Finance', 'Healthcare'",
    "education": "The highest level of education held by the candidate, e.g. 'Masters', 'Bachelors', 'PhD'",
    "university": "The name of the educational institution, e.g., 'University of Washington'",
    "certification": "Professional certification or license held, e.g., 'AWS Certified Data Analyst'",
    "project": "Notable project or research the candidate has worked on",
    "soft_skill": "Behavioral or interpersonal skill, e.g., 'leadership', 'teamwork', 'adaptability'",
    "experience_length": "Duration of work experience measured in years"
}

relation_types = {
    "hasSkill": "Candidate possesses this skill",
    "hasJobTitle": "Candidate has held this job title",
    "workedAt": "Candidate has worked at this company",
    "hasIndustry": "Candidate has worked in this industry",
    "studiedAt": "Candidate studied at this university",
    "certifiedIn": "Candidate holds this certification",
    "hasProject": "Candidate completed this project",
    "hasSoftSkill": "Candidate possesses this soft skill",
    "hasExperienceLength": "Candidate has this length of work experience",
    "hasEducationLevel": "Candidate attained this education level",
}

entity_relationship_match = {
    "skill": "hasSkill",
    "job_title":"hasJobTitle",
    "company": "workedAt",
    "industry": "hasIndustry",
    "education": "hasEducationLevel",
    "university": "studiedAt",
    "certification": "certifiedIn" ,
    "project": "hasProject",
    "soft_skill": "hasSoftSkill" ,
    "experience_length": "hasExperienceLength"
}

In [22]:
import json

system_prompt = f'''
    You are a helpful agent designed to fetch information from a graph database. 
    
    The graph database links candidates to the following entity types:
    {json.dumps(entity_types)}
    
    Each link has one of the following relationships:
    {json.dumps(relation_types)}

    Depending on the user prompt, determine if it possible to answer with the graph database.
        
    The graph database can match candidates with multiple relationships to several entities.
    
    Example user input:
    "Which candidate has 4 years of consulting experience?"
    
    There are two relationships to analyse:
    1. The mention of years means we will search for candidate with more than 4 years of experience
    2. The mention of experience means we will search for candidates with consulting experience
    
    
    Return a json object following the following rules:
    For each relationship to analyse, add a key value pair with the key being an exact match for one of the entity types provided, and the value being the value relevant to the user query.
    
    For the example provided, the expected output would be:
    {{
        "industry": "consulting",
        "experience_length": "4+ years",
        "age_group": "adults"
    }}
    
    If there are no relevant entities in the user prompt, return an empty json object.
'''

print(system_prompt)


    You are a helpful agent designed to fetch information from a graph database. 

    The graph database links candidates to the following entity types:
    {"skill": "A technical skill possessed by the candidate, e.g., 'Python', 'SQL', 'Tableau'", "job_title": "A specific role or position held by the candidate, e.g., 'Data Scientist', 'Software Engineer'", "company": "An organization where the candidate has worked", "industry": "The domain or sector associated with a company or job, e.g., 'Finance', 'Healthcare'", "education": "The highest level of education held by the candidate, e.g. 'Masters', 'Bachelors', 'PhD'", "university": "The name of the educational institution, e.g., 'University of Washington'", "certification": "Professional certification or license held, e.g., 'AWS Certified Data Analyst'", "project": "Notable project or research the candidate has worked on", "soft_skill": "Behavioral or interpersonal skill, e.g., 'leadership', 'teamwork', 'adaptability'", "experience_l

In [23]:
from openai import OpenAI
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", "<your OpenAI API key if not set as env var>"))

# Define the entities to look for
def define_query(prompt, model="gpt-4o"):
    completion = client.chat.completions.create(
        model=model,
        temperature=0,
        response_format= {
            "type": "json_object"
        },
    messages=[
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": prompt
        }
        ]
    )
    return completion.choices[0].message.content

In [24]:
example_queries = [
"Which candidate has 4 years of consulting experience?",
"Which candidate has studied at the University of Washington?",
"Which candidate has proficiency in SQL?", 
"Which candidate has an AWS certification?"
]

for q in example_queries:
    print(f"Q: '{q}'\n{define_query(q)}\n")


Q: 'Which candidate has 4 years of consulting experience?'
{
    "industry": "consulting",
    "experience_length": "4 years"
}

Q: 'Which candidate has studied at the University of Washington?'
{
    "university": "University of Washington"
}

Q: 'Which candidate has proficiency in SQL?'
{
    "skill": "SQL"
}

Q: 'Which candidate has an AWS certification?'
{
    "certification": "AWS"
}



### Generating queries

Now that we know what to look for, we can generate the corresponding Cypher queries to query our database. 

However, the entities extracted might not be an exact match with the data we have, so we will use the GDS cosine similarity function to return candidates that have relationships with entities similar to what the user is asking.

In [25]:
def create_embedding(text):
    result = client.embeddings.create(model=embeddings_model, input=text)
    return result.data[0].embedding

In [26]:
# The threshold defines how closely related words should be. Adjust the threshold to return more or less results
def create_query(text, threshold=0.5):
    query_data = json.loads(text)
    # Creating embeddings
    embeddings_data = []
    for key, val in query_data.items():
        if key != 'candidate':
            embeddings_data.append(f"${key}Embedding AS {key}Embedding")
    query = "WITH " + ",\n".join(e for e in embeddings_data)
    # Matching candidates to each entity
    query += "\nMATCH (p:Candidate)\nMATCH "
    match_data = []
    for key, val in query_data.items():
        if key != 'candidate':
            relationship = entity_relationship_match[key]
            match_data.append(f"(p)-[:{relationship}]->({key}Var:{key})")
    query += ",\n".join(e for e in match_data)
    similarity_data = []
    for key, val in query_data.items():
        if key != 'candidate':
            similarity_data.append(f"gds.similarity.cosine({key}Var.embedding, ${key}Embedding) > {threshold}")
    query += "\nWHERE "
    query += " AND ".join(e for e in similarity_data)
    query += "\nRETURN distinct p"
    return query

In [27]:
def query_graph(response):
    embeddingsParams = {}
    query = create_query(response)
    query_data = json.loads(response)
    for key, val in query_data.items():
        embeddingsParams[f"{key}Embedding"] = create_embedding(val)
    result = graph.query(query, params=embeddingsParams)
    return result

In [29]:
# example_queries = [
# "Which candidate has studied at the University of Washington and worked at Cisco Systems, Inc.",
# "Which candidate has proficiency in SQL?", 
# "Which candidate has an AWS certification?",
# "Which candidate has worked at Triumph group?",
# "Which candidate is suitable as a data scientist?",
# "Which candidate has analytics experience?"
# ]

example_queries = ["""
Responsibilities

The SCOT Labs team within Forecasting and Labs is responsible for designing and executing the inference and experimentation systems that measure the impact of SCOT initiatives. We are looking for senior applied scientists to drive innovation in SCOT by developing/building a new scientific approach and pushing our system further upstream in the innovation process. Key responsibilities of a Research Scientist in IPC Lab include:

 Developing new statistical, causal, and machine learning techniques and develop solution prototypes to drive innovation
 Working with technical and non-technical customers to design experiments and communicate results
 Collaborating with our dedicated software team to create production implementations for large-scale data analysis
 Developing an understanding of key business metrics / KPIs and providing clear, compelling analysis that shapes the direction of our business
 Presenting research results to our internal research community
 Leading training and informational sessions on our science and capabilities
 Your contributions will be seen and recognized broadly within Amazon, contributing to the Amazon research corpus and patent portfolio.

To help describe some of our challenges, we created a short video about at Amazon - http://bit.ly/amazon-scot

Amazon is an Equal Opportunity-Affirmative Action Employer – Minority / Female / Disability / Veteran / Gender Identity / Sexual Orientation / Age

Basic Qualifications

 3+ years of data querying languages (e.g. SQL), scripting languages (e.g. Python) or statistical/mathematical software (e.g. R, SAS, Matlab, etc.) experience
 3+ years of data scientist experience
 3+ years of machine learning/statistical modeling data analysis tools and techniques, and parameters that affect their performance experience
 Bachelor's degree
 Experience applying theoretical models in an applied environment

Preferred Qualifications

 Experience in Python, Perl, or another scripting language
 Experience in a ML or data scientist role with a large technology company
"""]

for q in example_queries:
    # print(f"Q: '{q}'\n{define_query(q)}\n")
    print("\n")
    result = query_graph(define_query(q))
    print(f"Found {len(result)} matching candidate(s):")
    for r in result:
      print(f"{r['p']['name']} ({r['p']['id']})")


Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownRelationshipTypeWarning} {category: UNRECOGNIZED} {title: The provided relationship type is not in the database.} {description: One of the relationship types in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing relationship type is: hasExperienceLength)} {position: line: 10, column: 7, offset: 375} for query: 'WITH $skillEmbedding AS skillEmbedding,\n$job_titleEmbedding AS job_titleEmbedding,\n$companyEmbedding AS companyEmbedding,\n$experience_lengthEmbedding AS experience_lengthEmbedding,\n$educationEmbedding AS educationEmbedding\nMATCH (p:Candidate)\nMATCH (p)-[:hasSkill]->(skillVar:skill),\n(p)-[:hasJobTitle]->(job_titleVar:job_title),\n(p)-[:workedAt]->(companyVar:company),\n(p)-[:hasExperienceLength]->(experience_lengthVar:experience_length),\n(p)-[:hasEdu

Found 0 matching candidate(s):
